In [15]:
import pandas as pd
import numpy as np
# Load master log-level dataset (defines correct row order)
df = pd.read_csv("../data/processed/uniswap_v3_usdc_weth_swaps_processed.csv")

# Load results (same row count as df)
if_results = pd.read_csv("../results/isolation_forest_results.csv")
ae_results = pd.read_csv("../results/autoencoder_results.csv")

# Normalize hash formatting
for d in (df, if_results, ae_results):
    d["transactionHash"] = d["transactionHash"].astype(str).str.strip().str.lower()

# Ensure row alignment (critical)
assert len(df) == len(if_results) == len(ae_results), "Row counts must match."

# Attach logIndex from df (log-level unique key)
if_results["logIndex"] = df["logIndex"].values
ae_results["logIndex"] = df["logIndex"].values

# Save clean log-level results
if_results.to_csv("../results/isolation_forest_results.csv", index=False)
ae_results.to_csv("../results/autoencoder_results.csv", index=False)

# Sanity: confirm alignment
assert (if_results["transactionHash"].values == ae_results["transactionHash"].values).all()

# Combine
results = pd.concat(
    [if_results, ae_results.drop(columns=["transactionHash"])],
    axis=1
)

results["if_anom"] = results["isolation_forest_anomaly"] == -1
results["ae_anom"] = results["ae_anomaly"] == -1

# Overlap categories
results["both_anom"] = results["if_anom"] & results["ae_anom"]
results["only_if"]   = results["if_anom"] & (~results["ae_anom"])
results["only_ae"]   = (~results["if_anom"]) & results["ae_anom"]
results["either"]    = results["if_anom"] | results["ae_anom"]

# Summary counts
summary = {
    "IF anomalies": int(results["if_anom"].sum()),
    "AE anomalies": int(results["ae_anom"].sum()),
    "Both anomalous": int(results["both_anom"].sum()),
    "IF only": int(results["only_if"].sum()),
    "AE only": int(results["only_ae"].sum()),
    "Union (either model)": int(results["either"].sum()),
}
print(summary)

# Crosstab (useful for the report)
print(pd.crosstab(results["if_anom"], results["ae_anom"], rownames=["IF"], colnames=["AE"]))


{'IF anomalies': 50, 'AE anomalies': 50, 'Both anomalous': 35, 'IF only': 15, 'AE only': 15, 'Union (either model)': 65}
AE     False  True 
IF                 
False    935     15
True      15     35


In [16]:
# Normalize scores to 0–1
iso = results["isolation_score"].astype(float)
ae  = results["ae_reconstruction_error"].astype(float)

results["iso_norm"] = (iso - iso.min()) / (iso.max() - iso.min() + 1e-9)
results["ae_norm"]  = (ae  - ae.min())  / (ae.max()  - ae.min()  + 1e-9)

# Fused risk score (equal weights as a neutral default)
results["risk_score"] = 0.5 * results["iso_norm"] + 0.5 * results["ae_norm"]

# Optional: define tiers by quantiles (stable and report-friendly)
q95 = results["risk_score"].quantile(0.95)
q99 = results["risk_score"].quantile(0.99)

def risk_tier(x):
    if x >= q99: return "HIGH"
    if x >= q95: return "MEDIUM"
    return "LOW"

results["risk_tier"] = results["risk_score"].apply(risk_tier)

print(results["risk_tier"].value_counts())


risk_tier
LOW       950
MEDIUM     40
HIGH       10
Name: count, dtype: int64


In [17]:
df = pd.read_csv("../data/processed/uniswap_v3_usdc_weth_swaps_processed.csv")

# Again, your pipeline keeps the same order
assert (df["transactionHash"].values == results["transactionHash"].values).all()

final = pd.concat([df.reset_index(drop=True), results.drop(columns=["transactionHash"]).reset_index(drop=True)], axis=1)

# Example: inspect top fused-risk transactions
cols = ["transactionHash","sender","recipient","amount0","amount1","liquidity","gasPrice","gasUsed",
        "isolation_score","ae_reconstruction_error","risk_score","risk_tier","if_anom","ae_anom","both_anom"]

final.sort_values("risk_score", ascending=False).head(20)[cols]


,transactionHash,sender,recipient,amount0,amount1,liquidity,gasPrice,gasUsed,isolation_score,ae_reconstruction_error,risk_score,risk_tier,if_anom,ae_anom,both_anom
901,0xa663daaf6615cc29397ad38f8fdfa3f7e7505c7f6a95...,0x3208684f96458c540eb08f6f01b9e9afb2b7d4f0,0x3208684f96458c540eb08f6f01b9e9afb2b7d4f0,-265430999843,161460000000000000000,834267773667859163307,0x2a05c99b7,0x13853f,0.147927,1495.304396,0.938582,HIGH,True,True,True
893,0x906c87dfc10add3947e88b32d7c8ab0982f93b77b6b7...,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,-223421442037,135900000000000000000,858490743562473777521,0x267115769,0x41d73,0.097589,1000.525081,0.713825,HIGH,True,True,True
828,0x9f1645ae3f3d8f808e31755c5dd0f89d3b13a273e2e8...,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,-221943584509,135000000000000000000,802204927987842757523,0x2c1e2c560,0x41d90,0.084263,1021.435996,0.705117,HIGH,True,True,True
31,0xd71237977566fc90f775c31715f7d443e46eb7e35672...,0xe8cfad4c75a5e1caf939fd80afcf837dde340a69,0xe8cfad4c75a5e1caf939fd80afcf837dde340a69,-523931437876,318093935332575067681,36436129027849465614,0x97c6c186fc,0x26f2a,0.175015,642.338974,0.685282,HIGH,True,True,True
63,0x47923e680c5b276593275e006e6c32d51ff8198d8bab...,0xe592427a0aece92de3edee1f18e0157c05861564,0xdef171fe48cf0115b1d80b88dc8eab59176fee57,100101107163,-60741857111288434426,1899816936058942905732,0x2e6a1c824,0x70593,0.123528,787.912601,0.673294,HIGH,True,True,True
889,0x4f19c6d7add8496b0f6c7816a20ec79f5fc1d223c4e6...,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,-222018850285,135000000000000000000,1808145905707358708007,0x2a5742805,0x4d55a,0.145237,695.040299,0.667818,HIGH,True,True,True
841,0x55870b25aef8b72d74d421928862506efbf550c416c0...,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,-209607817048,127500000000000000000,786091727607707560446,0x2f19e6ba4,0x49290,0.085051,894.502729,0.663600,HIGH,True,True,True
254,0x23b59448f8fc805b8f633bb15688af70b7290da9b8eb...,0xe592427a0aece92de3edee1f18e0157c05861564,0xdef171fe48cf0115b1d80b88dc8eab59176fee57,117900149524,-71663468374902938570,529443323562593035056,0x339fb2b1c,0x1b6dca,0.119940,412.362454,0.543489,HIGH,True,True,True
16,0x525bde5f069dac2457e3dccda54bdbef9bf268a3ac95...,0xe8cfad4c75a5e1caf939fd80afcf837dde340a69,0xe8cfad4c75a5e1caf939fd80afcf837dde340a69,-421800456347,255828679014122873255,42650691009184698025,0x38e5e5d377,0x1f9c4,0.200054,116.061900,0.538807,HIGH,True,True,True
824,0xa6ad3072341f593832ebe9875b95ad55239c76c91529...,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,0x3fc91a3afd70395cd496c647d5a6cc9d4b2b7fad,-246626742615,150000000000000000000,69023024792007499868,0x2ef852eb4,0x3148c,0.023677,662.103443,0.513576,HIGH,True,True,True


In [18]:
df = pd.read_csv("../data/processed/uniswap_v3_usdc_weth_swaps_processed.csv")

if_results = pd.read_csv("../results/isolation_forest_results.csv")
ae_results = pd.read_csv("../results/autoencoder_results.csv")

for d in (df, if_results, ae_results):
    d["transactionHash"] = d["transactionHash"].astype(str).str.strip().str.lower()

merged = (
    df.merge(if_results, on=["transactionHash","logIndex"], how="left")
      .merge(ae_results[["transactionHash","logIndex","ae_anomaly","ae_reconstruction_error"]],
             on=["transactionHash","logIndex"], how="left")
)

print("Merged rows:", len(merged))
print("Missing IF:", merged["isolation_forest_anomaly"].isna().sum())
print("Missing AE:", merged["ae_anomaly"].isna().sum())


Merged rows: 1000
Missing IF: 0
Missing AE: 0


In [19]:
# Safety check
required_cols = ["sender", "recipient", "agreement_group", "transactionHash"]
missing = [c for c in required_cols if c not in merged.columns]
assert not missing, f"Missing required columns: {missing}"

# Group by sender–recipient pairs
pair_report = (
    merged
    .groupby(["sender", "recipient"])
    .agg(
        total_tx=("transactionHash", "size"),
        both=("agreement_group", lambda s: (s == "BOTH").sum()),
        if_only=("agreement_group", lambda s: (s == "IF_ONLY").sum()),
        ae_only=("agreement_group", lambda s: (s == "AE_ONLY").sum()),
        neither=("agreement_group", lambda s: (s == "NEITHER").sum())
    )
    .reset_index()
)

# Total anomalies per pair
pair_report["anomaly_tx"] = (
    pair_report["both"] +
    pair_report["if_only"] +
    pair_report["ae_only"]
)

# Anomaly rate
pair_report["anomaly_rate"] = pair_report["anomaly_tx"] / pair_report["total_tx"]

# Optional: filter out very small pairs (recommended)
MIN_TX = 3
pair_report = pair_report[pair_report["total_tx"] >= MIN_TX]

# Sorting logic:
# 1) Highest anomaly rate
# 2) Highest anomaly count
# 3) Highest total activity
pair_report = pair_report.sort_values(
    ["anomaly_rate", "anomaly_tx", "total_tx"],
    ascending=False
)

# Clean formatting
pair_report["anomaly_rate"] = pair_report["anomaly_rate"].round(4)

# Show top pairs
pair_report.head(10)


AssertionError: Missing required columns: ['agreement_group']

In [ ]:
from sklearn.cluster import KMeans

# Select top 1% by fused risk
thr = merged["risk_score"].quantile(0.99)
high_risk = merged[merged["risk_score"] >= thr].copy()

print("High-risk events:", len(high_risk))

# Load scaled feature matrix
X_scaled = np.load("../data/processed/X_scaled.npy")

# Extract corresponding rows
X_high = X_scaled[high_risk.index]

# Cluster if possible
if len(high_risk) >= 3:
    kmeans = KMeans(n_clusters=2, random_state=42, n_init="auto")
    high_risk["cluster"] = kmeans.fit_predict(X_high)
else:
    high_risk["cluster"] = 0

print("\nCluster counts:")
print(high_risk["cluster"].value_counts())

# Cluster profiles
with open("../data/processed/feature_names.txt") as f:
    features = [line.strip() for line in f]

profiles = pd.DataFrame(X_high, columns=features).groupby(high_risk["cluster"]).mean()
print("\nCluster profiles:")
print(profiles)


KeyError: 'risk_score'